In [1]:
import pandas as pd
import re
import numpy as np
from datetime import datetime
from dateutil import parser
from Levenshtein import ratio

In [2]:
def open_textfile(image_path: str) -> dict:
    data_dict = {}
    with open(image_path, "r") as f:
        lines = f.readlines()
        for line in lines:
            try:
                pattern = r"(?<=\d):"  # Match ':' only if preceded by a digit
                key, value = re.split(pattern, line.strip())
                data_dict[key] = value
            except Exception as e:
                return f"Error reading line in file: {line}: {e}" 
    return data_dict

In [3]:
# focus on evaluating taxon, geography, and date 

# event dates ground truth labels
#dates_path = "transcription/data/new-england-samples/output/dates.txt"
dates_path = "/Users/mvoong/Desktop/spark-symbiota-ml/transcription/data/new-england-samples/output/dates.txt"
dates_dict = open_textfile(dates_path)

# geography ground truth labels 
geography_path = "/Users/mvoong/Desktop/spark-symbiota-ml/transcription/data/new-england-samples/output/localities.txt"
geography_dict = open_textfile(geography_path)

# taxon ground truth labels
taxon_path = "/Users/mvoong/Desktop/spark-symbiota-ml/transcription/data/new-england-samples/output/taxons.txt"
taxon_dict = open_textfile(taxon_path)

In [4]:
results = pd.read_csv("/Users/mvoong/Desktop/spark-symbiota-ml/transcription/results/azure_doc_intell_results_0.csv", index_col=False)

In [5]:
results.head()

,recordedBy,location,scientificName,eventDate,barcode,institutionCode,image_path
0,J. J. Davis,"East Greenwich, R. I.",Kalmia latifolia,"July 4, 787",369332,WISCONSIN STATE HERBARIUM (WIS),./data/new-england-samples/output/5077536349.jpeg
1,Leston A. Wheeler,"Windham, VT",Asclepias zynaca,3-Jul-14,09-016,University of South Florida Herbarium,./data/new-england-samples/output/1503302652.jpeg
2,"C. F. Jenney, J. R. Churchill, A. F. Hill","Wet meadow near Burnt Head, Lincoln County, Maine",Osmunda cimamonia L.,1-Jul-19,YU.008327,Yale University Herbarium,./data/new-england-samples/output/1038983765.jpeg
3,C. H. Biss,"Southington, Connecticut",Hieracium floribundum,5-Jun-04,CBS.35260,Connecticut Botanical Society,./data/new-england-samples/output/1038919101.jpeg
4,H. S. Clark,"North Woodbury, Connecticut",Viola rotundifolia Michx.,23-Jul-05,CBS.18531,Connecticut Botanical Society,./data/new-england-samples/output/1038940640.jpeg


In [6]:
for _, row in results.iterrows():
    #pred_collector = row["collector"]
    #pred_date = row["eventDate"]
    pred_geography = row["location"]
    pred_taxon = row["scientificName"]
    pred_taxon_processed = " ".join(pred_taxon.split(" ")[:2])
    occid = str(row["image_path"].split("/")[-1].split(".jpeg")[0])
    #occid = str(row["occurrence_id"])
    
    try:
        #actual_collector = collector_dict[occid]
        #actual_date = dates_dict[occid]
        actual_geography = geography_dict[occid]
        actual_taxon = taxon_dict[occid]
    except Exception as e:
        print(occid)

4850289459


In [7]:
def get_best_candidate(predicted_values: list, actual_value):
    if predicted_values:
        # compute levenshtein ratio between the actual value and each value in the predicted values list 
        final_pred_value, value_ratio_max = max([(pred_v, ratio(actual_value.lower(), pred_v.lower())) for pred_v in predicted_values], 
                                                  key=lambda x: x[1])
        return final_pred_value, value_ratio_max
    
    return predicted_values, 0  

In [8]:
def format_date_string(date_str: str) -> str:
    # format extracted date strings to match ground truth 

    date_str = date_str.strip()

    # check if the date is only month and year
    if re.fullmatch(r"[A-Za-z]+ \d{4}", date_str):
        try:
            dt = parser.parse(date_str)
            return dt.strftime("%Y-%m")
        except Exception:
            return date_str
    else:
        try:
            dt = parser.parse(date_str, dayfirst=True)
            return dt.strftime("%Y-%m-%d")
        except (ValueError, TypeError):
            return date_str

In [9]:
def append_results(results_df: pd.DataFrame) -> pd.DataFrame:
    lev_ratio_taxons = []
    lev_ratio_geography = []
    lev_ratio_dates = []

    gt_taxons = []
    gt_geography = []
    gt_dates = []
    pred_taxons_cleaned = []
    pred_dates_cleaned = []

    for _, row in results_df.iterrows():
        # get predicted values
        pred_taxon = row["scientificName"]
        pred_geography = row["location"]
        pred_date = row["eventDate"]
        
        # clean predicted taxon name for comparing to ground truth, get genus species
        if ";" in pred_taxon:
            pred_taxon_processed = "".join(pred_taxon.split(";")[-1])
        else:
            pred_taxon_processed = " ".join(pred_taxon.split(" ")[:2])
        pred_taxons_cleaned.append(pred_taxon_processed)

        # id 
        occid = str(row["image_path"].split("/")[-1].split(".jpeg")[0])

        # get actual values 
        try:
            actual_taxon = taxon_dict[occid]
            gt_taxons.append(actual_taxon)

            actual_geography = geography_dict[occid]
            gt_geography.append(actual_geography)

            actual_date = dates_dict[occid]
            gt_dates.append(actual_date)

        except KeyError as ke:
            gt_taxons.append("") #Keyerror 
            gt_geography.append("") #Keyerror
            gt_dates.append("") #Keyerror

            print(f"key errror on {occid}")

        # compute levenshtein ratio between the predicted value and ground truth 
        lev_ratio_taxons.append(ratio(actual_taxon.lower(), pred_taxon_processed.lower()))
        lev_ratio_geography.append(ratio(actual_geography.lower(), pred_geography.lower()))
        
        pred_date_cleaned = format_date_string(pred_date)
        pred_dates_cleaned.append(pred_date_cleaned)

        lev_ratio_dates.append(ratio(actual_date, pred_date_cleaned))

    # append actual taxon to the dataframe 
    results_df["Actual Taxon"] = gt_taxons

    # append cleaned predicted taxon to the dataframe 
    results_df["Cleaned Predicted Taxon"] = pred_taxons_cleaned

    # append similarity scores to the dataframe
    #results_df["Cosine_Similarity_taxon"] = cosine_sim_taxons
    results_df["Lev Ratio Taxon"] = lev_ratio_taxons

    results_df["Actual Geography"] = gt_geography
    #results_df["Cosine_Similarity_geo"] = cosine_sim_geography
    results_df["Lev Ratio Geo"] = lev_ratio_geography

    results_df["Actual Date"] = gt_dates
    results_df["Cleaned Predicted Date"] = pred_dates_cleaned
    #results_df["Cosine_Similarity_date"] = cosine_sim_dates

    results_df["Lev Ratio date"] = lev_ratio_dates 
    return results_df

In [10]:
def calculate_taxon_accuracy(results_df: pd.DataFrame, sim_threshold=0.8) -> float:
    acc_count = (results_df["Lev Ratio Taxon"] > sim_threshold).sum()  
    acc_val = acc_count/len(results_df)
    return acc_val*100, acc_count

def calculate_geo_accuracy(results_df: pd.DataFrame, sim_threshold=0.8) -> float:
    acc_count = (results_df["Lev Ratio Geo"] > sim_threshold).sum()  
    acc_val = acc_count/len(results_df)
    return acc_val*100, acc_count

def calculate_date_accuracy(results_df: pd.DataFrame, sim_threshold=0.8) -> float:
    acc_count = (results_df["Lev Ratio date"] > sim_threshold).sum()  
    acc_val = acc_count/len(results_df)
    return acc_val*100, acc_count

In [11]:
azure_df = append_results(results)
azure_tax, azure_count_tax =  calculate_taxon_accuracy(azure_df, 0.7)
azure_geo, azure_count_geo =  calculate_geo_accuracy(azure_df, 0.7)
azure_date, azure_count_date =  calculate_date_accuracy(azure_df, 0.7)

#len(azure_df[azure_df['Lev Ratio Taxon'] > 0.7])
print(f"taxon accuracy: {azure_tax}")
print(f"geo accuracy: {azure_geo}")
print(f"date accuracy: {azure_date}")

key errror on 4850289459
taxon accuracy: 74.0
geo accuracy: 10.0
date accuracy: 72.0


In [12]:
def analyze_counts(results_df: pd.DataFrame, col_name: str):
    values = [0, 0.2, 0.5, 0.7, 0.9]

    counts = []
    for i in range(len(values)-1):
        lower = values[i]
        upper = values[i+1]
        count = len(results_df[(results_df[col_name] > lower) & (results_df[col_name] < upper)])
        counts.append(count)

    count_last = len(results_df[results_df[col_name] > 0.9])
    counts.append(count_last)
    return counts

In [13]:
def display_counts(counts_list: list):
    x_labels = ['>0 & <0.2', '>0.2 & <0.5', '>0.5 & <0.7', '>0.7 & <0.9', '>0.9']

    i = 0 
    for count in counts_list:
        print(f"threshold: {x_labels[i]}, count: {count}")
        i+=1

In [14]:
azure_taxon_counts = analyze_counts(azure_df, 'Lev Ratio Taxon')
azure_geo_counts = analyze_counts(azure_df, 'Lev Ratio Geo')
azure_date_counts = analyze_counts(azure_df, 'Lev Ratio date')

In [15]:
print("taxon")
display_counts(azure_taxon_counts)
print("geo")
display_counts(azure_geo_counts)
print("date")
display_counts(azure_date_counts)

taxon
threshold: >0 & <0.2, count: 0
threshold: >0.2 & <0.5, count: 2
threshold: >0.5 & <0.7, count: 11
threshold: >0.7 & <0.9, count: 11
threshold: >0.9, count: 26
geo
threshold: >0 & <0.2, count: 1
threshold: >0.2 & <0.5, count: 28
threshold: >0.5 & <0.7, count: 13
threshold: >0.7 & <0.9, count: 3
threshold: >0.9, count: 2
date
threshold: >0 & <0.2, count: 1
threshold: >0.2 & <0.5, count: 5
threshold: >0.5 & <0.7, count: 5
threshold: >0.7 & <0.9, count: 26
threshold: >0.9, count: 10


In [16]:
google_results = pd.read_csv("/Users/mvoong/Desktop/spark-symbiota-ml/transcription/results/google_doc_ai_results.csv", index_col=False)
google_df = append_results(google_results)
google_tax, google_count_tax = calculate_taxon_accuracy(google_df, 0.7)
google_geo, google_count_geo = calculate_geo_accuracy(google_df, 0.7)
google_date, google_count_date = calculate_date_accuracy(google_df, 0.7)

print(f"taxon accuracy: {google_tax}")
print(f"geo accuracy: {google_geo}")
print(f"date accuracy: {google_date}")

key errror on 4850289459
taxon accuracy: 72.0
geo accuracy: 14.000000000000002
date accuracy: 70.0


In [17]:
google_taxon_counts = analyze_counts(google_df, 'Lev Ratio Taxon')
google_geo_counts = analyze_counts(google_df, 'Lev Ratio Geo')
google_date_counts = analyze_counts(google_df, 'Lev Ratio date')

In [18]:
print("taxon")
display_counts(google_taxon_counts)
print("geo")
display_counts(google_geo_counts)
print("date")
display_counts(google_date_counts)

taxon
threshold: >0 & <0.2, count: 1
threshold: >0.2 & <0.5, count: 4
threshold: >0.5 & <0.7, count: 9
threshold: >0.7 & <0.9, count: 8
threshold: >0.9, count: 28
geo
threshold: >0 & <0.2, count: 2
threshold: >0.2 & <0.5, count: 23
threshold: >0.5 & <0.7, count: 16
threshold: >0.7 & <0.9, count: 5
threshold: >0.9, count: 2
date
threshold: >0 & <0.2, count: 1
threshold: >0.2 & <0.5, count: 5
threshold: >0.5 & <0.7, count: 8
threshold: >0.7 & <0.9, count: 19
threshold: >0.9, count: 16


In [19]:
len(azure_df[azure_df['Lev Ratio Taxon'] > 0.7])

37

In [20]:
google_df[google_df['Lev Ratio Taxon'] < 0.7]

,recordedBy,location,scientificName,eventDate,barcode,institutionCode,image_path,Actual Taxon,Cleaned Predicted Taxon,Lev Ratio Taxon,Actual Geography,Lev Ratio Geo,Actual Date,Cleaned Predicted Date,Lev Ratio date
3,C. H. Bissell,"Frisbie farm, Queen St, Southington, Connecticut",H. pratense; Theracion floribundum,5-Jun-04,CBS.35260,Connecticut Botanical Society,./data/new-england-samples/output/1038919101.jpeg,Pilosella caespitosa,Theracion floribundum,0.279070,Frisbie farm Gwen St,0.521739,1904-06-05,2004-06-05,0.761905
10,A. W. Cheever,"New Boston, Hillsboro County, New Hampshire",Carex abdita Bickn.,15-Jun-03,65764,University of South Florida,./data/new-england-samples/output/1978899246.jpeg,Carex umbellata,Carex abdita,0.642857,New Boston.,0.363636,1903-06-15,2003-06-15,0.761905
13,H. G. Jesup,"Amherst, Massachusetts",Salix cordata,"May, 1872",6692,The Field Museum,./data/new-england-samples/output/1228599514.jpeg,Salix eriocephala,Salix cordata,0.580645,Amherst,0.466667,1872-05,1872-05-07,0.777778
14,G. G. Kennedy,"Purgatory Swamp, Norwood",Carex laxiflora Lam.,15 June 1895,34820,University of South Florida Herbarium,./data/new-england-samples/output/1947780920.jpeg,Carex digitalis,Carex laxiflora,0.516129,Purgatory Swamp; Norwood.,0.920000,1895-06-15,1895-06-15,0.952381
20,G. E. Nichols,"Bog- nnfolk, Connecticut",Carex paupercula Michx.,Jun. 13/12,YU.037642,Herbarium of Yale University,./data/new-england-samples/output/1039015537.jpeg,Carex magellanica,Carex paupercula,0.588235,Bog,0.214286,1912-06-13,2012-06-13,0.761905
22,Walter Deane,UNKNOWN,UNKNOWN,3 August 1889,UNKNOWN,The Field Museum,./data/new-england-samples/output/1228180607.jpeg,Solidago juncea,UNKNOWN,0.173913,Jaffrey,0.000000,1889-08-03,1889-08-03,0.952381
27,Henry B. Gilson,"Grafton County, NH",Actaea spicata (L.) Willd.,31 May 1899,NHA-550830,University of New Hampshire,./data/new-england-samples/output/5067933290.jpeg,Actaea rubra,Actaea spicata,0.592593,[no precise locality],0.300000,1899-05-31,1899-05-31,0.952381
28,C. H. Bissell,"Connecticut, along Willimantic River, Thundham",Linaria cymbalaria,2-Jul-25,CBS.29819,Connecticut Botanical Society,./data/new-england-samples/output/1039016550.jpeg,Cymbalaria muralis,Linaria cymbalaria,0.594595,Waste ground along Willimantic River,0.626506,1902-07-08,2025-07-02,0.666667
31,Willard Sherman Turrell,"Groton, New Hampshire",Lycopodium annotinum var. pungens,Sep-82,MU 000105005,Oberlin College Herbarium,./data/new-england-samples/output/4850289459.jpeg,,Lycopodium annotinum,0.270270,,0.279070,,1982-09-07,0.476190
35,Daniel C. Eaton,"Westfield Woods, New Haven, Connecticut",Habenaria hookeri Torrey,"June 7, 1867",YU.029479,Yale University Herbarium,./data/new-england-samples/output/1038946733.jpeg,Platanthera hookeri,Habenaria hookeri,0.648649,Westfield Woods,0.545455,1867-06-07,1867-06-07,0.952381


In [21]:
from wcvpy.wcvp_name_matching import get_accepted_info_from_names_in_column

your_data_df = google_df
name_col = 'Cleaned Predicted Taxon'  # Name of column in data with names to check
match_level = 'full'
data_with_accepted_information = get_accepted_info_from_names_in_column(your_data_df, name_col, match_level=match_level)

Loading WCVP locally if exists...
from: /Users/mvoong/.wcvp_downloads/wcvp.zip
A new checklist version was released at: 2026-01-07 07:36:01-05:00
To up date the WCVP version, run get_all_taxa(get_new_version=True)
Parsing the checklist
Time elapsed for (down)loading WCVP: 25.320481061935425s


/opt/anaconda3/envs/herbariaocr/lib/python3.13/site-packages/wcvpy/wcvp_name_matching/knms_name_matching.py:94: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  records['submitted'].ffill(inplace=True)
/opt/anaconda3/envs/herbariaocr/lib/python3.13/site-packages/wcvpy/wcvp_name_matching/knms_name_matching.py:95: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the int

All KNMS records return false. Not saving these records as this sometimes indicates server issues.


/opt/anaconda3/envs/herbariaocr/lib/python3.13/site-packages/wcvpy/wcvp_name_matching/general_matching_utils.py:91: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  match_df[id_col_name] = match_df[id_col_name].replace('no_id_placeholder', np.nan)


Trying to resolve 6 names with OpenRefine


/opt/anaconda3/envs/herbariaocr/lib/python3.13/site-packages/wcvpy/wcvp_name_matching/general_matching_utils.py:91: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  match_df[id_col_name] = match_df[id_col_name].replace('no_id_placeholder', np.nan)


Resolving submitted names which weren't initially matched using KNMS..
Check tempfile: name matching temp outputs/unmatched_to_autoresolveddf83c3f109e03369b506a5bb5e6eb3e.csv.
This may take some time... This can be sped up by specifying families of interest (if you haven't already done so) or checking the temp file for misspelled submissions.


Searching automated matches: 100%|██████| 7/7 [00:00<00:00, 5184.55it/s]


Check tempfile: name matching temp outputs/unmatched_samplesddf83c3f109e03369b506a5bb5e6eb3e.csv.
Temp file for this run:
name matching temp outputs/final_resolutions7ac3f3102db6ab5b311e671cd51d61c8.csv


In [22]:
# Update Cleaned Predicted Taxon if it doesn't match accepted_species
original_cleaned = data_with_accepted_information['Cleaned Predicted Taxon'].copy()

data_with_accepted_information['Cleaned Predicted Taxon_updated'] = data_with_accepted_information.apply(
    lambda row: row['accepted_species'] if pd.notna(row['accepted_species']) and row['Cleaned Predicted Taxon'] != row['accepted_species'] else row['Cleaned Predicted Taxon'],
    axis=1
)

changed_count = (original_cleaned != data_with_accepted_information['Cleaned Predicted Taxon_updated']).sum()
print(f"Number of rows changed: {changed_count}")

Number of rows changed: 12


In [23]:
# Calculate new Levenshtein ratio for Actual Taxon and Cleaned Predicted Taxon_updated
data_with_accepted_information['Lev Ratio Taxon Updated'] = data_with_accepted_information.apply(
    lambda row: ratio(row['Actual Taxon'].lower(), row['Cleaned Predicted Taxon_updated'].lower()) if row['Actual Taxon'] and row['Cleaned Predicted Taxon_updated'] else 0,
    axis=1
)

In [24]:
len(data_with_accepted_information[data_with_accepted_information['Lev Ratio Taxon Updated'] > 0.71])/50

0.82

In [25]:
az_your_data_df = azure_df
az_data_with_accepted_information = get_accepted_info_from_names_in_column(az_your_data_df, name_col, match_level=match_level)

Loading WCVP locally if exists...
from: /Users/mvoong/.wcvp_downloads/wcvp.zip
A new checklist version was released at: 2026-01-07 07:36:01-05:00
To up date the WCVP version, run get_all_taxa(get_new_version=True)
Parsing the checklist
Time elapsed for (down)loading WCVP: 23.70952606201172s


/opt/anaconda3/envs/herbariaocr/lib/python3.13/site-packages/wcvpy/wcvp_name_matching/knms_name_matching.py:94: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  records['submitted'].ffill(inplace=True)
/opt/anaconda3/envs/herbariaocr/lib/python3.13/site-packages/wcvpy/wcvp_name_matching/knms_name_matching.py:95: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the int

All KNMS records return false. Not saving these records as this sometimes indicates server issues.


/opt/anaconda3/envs/herbariaocr/lib/python3.13/site-packages/wcvpy/wcvp_name_matching/general_matching_utils.py:91: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  match_df[id_col_name] = match_df[id_col_name].replace('no_id_placeholder', np.nan)


Trying to resolve 20 names with OpenRefine
Resolving submitted names which weren't initially matched using KNMS..
Check tempfile: name matching temp outputs/unmatched_to_autoresolve13d22f10e6d9cb6a7d621c68819c2543.csv.
This may take some time... This can be sped up by specifying families of interest (if you haven't already done so) or checking the temp file for misspelled submissions.


Searching automated matches: 100%|█████| 10/10 [00:00<00:00, 201.17it/s]


Check tempfile: name matching temp outputs/unmatched_samplesad622dc9a55d5b041b59056f3cb12722.csv.
Temp file for this run:
name matching temp outputs/final_resolutions1e1abd41ec44fc2ae18dd7200096a3a9.csv


In [26]:
# Update Cleaned Predicted Taxon if it doesn't match accepted_species
az_original_cleaned = az_data_with_accepted_information['Cleaned Predicted Taxon'].copy()

az_data_with_accepted_information['Cleaned Predicted Taxon_updated'] = az_data_with_accepted_information.apply(
    lambda row: row['accepted_species'] if pd.notna(row['accepted_species']) and row['Cleaned Predicted Taxon'] != row['accepted_species'] else row['Cleaned Predicted Taxon'],
    axis=1
)

az_changed_count = (az_original_cleaned != az_data_with_accepted_information['Cleaned Predicted Taxon_updated']).sum()
print(f"Number of rows changed: {az_changed_count}")

Number of rows changed: 11


In [27]:
# Calculate new Levenshtein ratio for Actual Taxon and Cleaned Predicted Taxon_updated
az_data_with_accepted_information['Lev Ratio Taxon Updated'] = az_data_with_accepted_information.apply(
    lambda row: ratio(row['Actual Taxon'].lower(), row['Cleaned Predicted Taxon_updated'].lower()) if row['Actual Taxon'] and row['Cleaned Predicted Taxon_updated'] else 0,
    axis=1
)

In [28]:
len(az_data_with_accepted_information[az_data_with_accepted_information['Lev Ratio Taxon Updated'] > 0.71])/50

0.78

# updated location counts with post processing

In [29]:
# sometimes the predicted geography is more verbose than the actual location as logged in gbif 
# if the levenshtein ratio is less than 0.7 
# check if the actual geography is in the location if so then consider it a match 
updates = 0
for _, row in google_results.iterrows():
    if row["Lev Ratio Geo"] < 0.7 and row["Actual Geography"].strip() != "[no precise locality]" :
        pred_geo = row["location"]
        actual_geo = row["Actual Geography"]
        if actual_geo.split(".")[0].strip() in pred_geo.strip() and actual_geo.split(".")[0].strip():
            row["Lev Ratio Geo Updated"] = 1.0 
            updates+=1
            print(actual_geo.split(".")[0].strip())

East Greenwich
North Woodbury
Pine Plains
Middletown
New Boston
Amherst
Moose Hill
Bog
Charlotte
Peterboro
Goshen Pond meadows
Westfield Woods
West Pond
Orange
Stepney
Middlebury


In [30]:
(updates + len(google_df[google_df["Lev Ratio Geo"] > 0.7]))/50

0.46

In [31]:
updates_az = 0
for _, row in results.iterrows():
    if row["Lev Ratio Geo"] < 0.7 and row["Actual Geography"].strip() != "[no precise locality]" :
        pred_geo = row["location"]
        actual_geo = row["Actual Geography"]
        if actual_geo.split(".")[0].strip() in pred_geo.strip() and actual_geo.split(".")[0].strip():
            row["Lev Ratio Geo Updated"] = 1.0 
            updates_az+=1
            print(actual_geo.split(".")[0].strip())

East Greenwich
North Woodbury
New Boston
Bog
Charlotte
Peterboro
Westfield Woods
West Pond
Orange
Stepney
Middlebury


In [32]:
(updates_az + len(azure_df[azure_df["Lev Ratio Geo"] > 0.7]))/50

0.32

In [54]:
print(len(data_with_accepted_information[data_with_accepted_information["Lev Ratio Taxon Updated"] > .71])/50)

0.82


In [55]:
len(az_data_with_accepted_information[az_data_with_accepted_information['Lev Ratio Taxon Updated'] > .71])/50

0.78

In [52]:
data_with_accepted_information.to_csv("/Users/mvoong/Desktop/spark-symbiota-ml/transcription/results/google_doc_ai_results_appended.csv")

In [53]:
az_data_with_accepted_information.to_csv("/Users/mvoong/Desktop/spark-symbiota-ml/transcription/results/azure_doc_intell_results_appended.csv")